# Hello Kaggle Community! 👋
# 🐈‍⬛ Black Cat SWE Agent | Pack Instinct · v12: The State Card

The v10 bundle scored **0.12** on the public leaderboard. v11 added the `repo-scout` skill and passed official compilation plus a small GPU smoke test; its leaderboard score is still unknown. v12 is an **experimental challenger** built from that exact v11 bundle. It adds a concise working-state protocol and a conditional second diagnosis after a failed repair, with a five-minute task limit.

| Bundle | Evidence | Role |
|---|---|---|
| v10 | Public score **0.12** | Scored anchor, preserved byte for byte |
| v11 | Official compile and two Rich smoke tasks passed | Engineering parent, score unknown |
| v12 | Static checks and clean notebook run below | Experimental submission; score unknown |

**Hypothesis.** A short state card will help the coder retain the confirmed location, failed hypothesis and next action when long tool outputs leave context. One conditional second diagnosis may prevent an edit loop. These are hypotheses, not observed gains. The public/private leaderboard relationship is unknown.

In [ ]:
from pathlib import Path
import base64, hashlib, io, json, platform, posixpath, re, zipfile
import yaml

WORKING = Path('/kaggle/working') if platform.system() == 'Linux' and Path('/kaggle/working').exists() else Path.cwd()
WORKING.mkdir(parents=True, exist_ok=True)
print('🐈‍⬛ Pack Instinct v12 | Python', platform.python_version(), '| output:', WORKING)

## 1 · Pin the scored anchor and engineering parent

All three input archives are embedded so this notebook runs without internet or an external dataset. Hash checks catch accidental changes before any challenger is constructed. The fallback is the v6 bundle, which previously scored 0.10 twice.

In [ ]:
ARCHIVE_B64 = {'v11': 'UEsDBBQAAAAIAAAAIQD6opVKHAEAANoBAAAKAAAAYWdlbnQueWFtbE1QSW7DMAy86xXsvUoQNOjBt77EYCzaIarFFam6zusrWejig6yhhpglYqABZKNxSo6yCfX0AywUAtqrfbncLKv9QLXbFS+vdlLjSKbMq3KKA7wVTTGFVAQkzbphJqC4cCTKoHdUmPmLBDKtSVhT3oFFSp1srHcIHDmgf4ZPyjwzOVhRpzvJyXAUzWXqMk8cJ18cwZpTWFXOsotSOAVnFoqUUVuAqBS1/Wde/u30Qd3BsHqOy2nH4I2m5GUwABZyiXWrJo6uY0I3zuzpQORY/9CWuWr9woV0FEUtckApt1DZR4hjgEuz1LSaVPu6m0a5D40/HhQ5t/5HjOj3B+VusS/IO6+jlGov8wN7H7UZMvXB/0To93Or2cqUippvUEsDBBQAAAAIAAAAIQASrgpEawAAAIIAAAAVAAAAY29uZmlncy9zYW1wbGluZy55YW1sPYxBDoMwDATveYVfUNGqVE0+Y6VgkihgR2BLfX4Dh95Go51V2hrtUW2nAMPt4VQathP9eHEN8BzcFr8ops0UVSrxEeB9932dC9fCCSfhpaTgAP7qY3MiPXP/6r7wtNpMqFksZe0PS1wPcj9QSwMEFAAAAAgAAAAhAH7UgdxNAAAAYQAAABAAAABldmFsX2NvbmZpZy55YW1sSy1LzClNLMnMz7PiUlAoycxNzS8tiS9OTc7PSym2UjA2MAAK5yZWxJfk5+fEJyfm5ABFDRGiQA3xuZl5pSWpYHGYcGlRHpBvZGrABQBQSwMEFAAAAAgAAAAhANJpY6pJAwAAqQUAABMAAABwcm9tcHRzL2FuYWx5emVyLm1kXVRNb+M2EL3rVwx2L7uArW236CUIDBiOtwiQ2kXsAs3JoqSxRIQiXQ4V20V/fN9Q2Y/0YIMi5+O9N498CiOZyFQ1oeWD8cZd/+FYzchQZNPOg3dX0jPy5sV2JtngSU7cWOOspJKeUMDzC0caQmuPVzpax1IWv9kX9mQ8WZGRZ9j2LfHFNAkFzz2jp000jJKoZpxeuC2L4v172ofgpJjTvgeqyKcwlyaMqSJ5ts7NaBRuER8l3eB89Ie8f5Am2hOizjb1U+jBm+FtiVkGdzgZhFRTgnxyoTGJy9O1AtoWYnRC1d8jR2wIJ0qBEqBkGpRsckwnNwrQCz3zlWzLPtmj5Si5AMcYIiW+QJz7RNH4Z8kVhgCukR2/GJ/oOPpGxZySsmhkIVgNoBBfuAm+Lb/DlD6cM8hMcEKpBCs6ReuBxVDLENnmCeUgZz3GNg41oJVQNKvVhGFAx4qOQPm4Xt7Nt5uHJ3rdFtKBQ9gOstE8emhWOdF/lX2evzsMzoWO5ojYVVpY2MSmP4gdrDPxoH6ZGoBEw4D/g4QtK6UadBWksgUN67vsMi3WccoVDp5t19chSqVDOBv3TI1x7qvQec3yJkXGuovm1OcMYSaoBmWOfEa4b22LUZNchxoeU3Cem5SVgdcPOoRXfRM6p0lADLADWNRDPHw3ZJf+zsDeFj+XtL6kCFe/8cExhuE75Ztvw/7UOCOihFlmr07BUkyn39qf1JxYh1M2R1l8LmmX1c16ssEieNwnVPfYcg4EtZOqQU1v4KEROBxdcTFjjtfjTGW6djX3uMphjNTijkYll/Gee5N+mBNfcMsTEPxS0mpiTvU1vwo6LQ0E7dG4PLiSNvkR6EbQ+Z/zVK+llzOOwWFAlw/45evw+def6BxiC8Y+pF4LsxP+WDxsV8v9/XZzQ7eqyOLmVpKJaTG/Zd8u6MPtV0lJXaaqLj4Wj9vtnlbLP3drpEElPUtnNYJPDCPKovhy/xf98bDUuurNyPADVMOIF8Xj+mG5X99pLtjFSVKxCQKh0HRFPfM3+qLvy5Q804h3Hj3fLYr9erffoQpf8ELmYAbVKT+pyHzh2FjBc9JbyfItitV28+X+br1ZAXsP99G/cEZrxwELzLj4D1BLAwQUAAAACAAAACEAtpZioNwFAADnCgAAEQAAAHByb21wdHMvc3lzdGVtLm1khVbbbtw2EH3frxg0KJIsVnKbNEXhpgG2rtMYdW3Xdpr2acWVRivCEqmQlHcVFH3sB/QT+yU9Q2p96UtfFiuJnDlz5swhf7cDKcekDKkhWGM7O3jybLR1dDGGxhpis9GG2dHWuhttNqSN1xX2kFemWtsdV1Q2XN7YIZCt8d6xasn2bDJvB1cyXvTW62DdSCrQgQTyvSo5n/1oVXuI7962t0yhYdLeD/g18WHwyNux92rDtNWhiW99p9qWfaDSOsdloF6FslnIN0MlvlHhh3Wnwyp+KPLZ7MkTeqdcRW7AxllGZ3yLyFzpsCBVVYRyK245AAMC+wUVpTW1/M/7scBjP8YHbTSejk5kAyq4URthpNaImtM7XVVAECMkWvu+1aBHIZKjEWTXCOCbHAh+Yu6pH9atLml5ceJpjXBbwVjaDrj1ukX9BoH9A154h4ilDi2Y9DeeauBQoF+ZDUvUq9JJyQkRbSwFSweh68madsxpadBTQdxyHYTk+17QmpEYm3rlYh8B1yVmJfA1ILC51c6ajg32YqHjbD3oNhxSZcnYQAENRkIIJEgXEkFgBvuX7VaNfqqf1mPskyD5T6toiS+O66EFIB8yrlFiwL4dnhV4NXYPSpr6AfDr1m5nX+Y0n783FTvkNtV8fkj4E5KmwBpkgkbcokuqDIOS6I261VIjEEutnttaKLGGpbtha2UQAptSKngh8U8tUOtPjOgzIsroSMqUDJBLxStlVDt+Ylcgpm3vBYtq2qmBgXdSDVDmdBKg/DA44+n0/Gh5fXJ+Rgd0eX5+TUfL91fHeHh78htdnC7PcvqVna5H0qCgbJXuhEMMWiUcYk69VInSCKSiiWsGbRwFjgV5Qnu8C06WiPYRCVoNutbsFsTO2ftBA39RQGRUB/DOdg8UKB89KyciE5oa7g6p2GDEKXOGPnt9H/fNZ5Rl2pTtUPF3cwwS5fQHNQBN2csvignVpTyLPGOSBF9GBZ5T0bNCalwJmiLxqeIScqJ46VPhsSwz9PTqenl5vTg++6F/CtpOj4vn+eyldO2Se2eroZS20dbpIN7VQYiwEfKl033Yj8mBk7WCNDRwKt/YbZq+9bCJlbvBoAUJiXiCGOSjjajqK0n6Vu8knTSAJhdMM5m2yvtUVU5HFvkK21YrHxy6VRD6s4YDdPfcy9IFzROZaYBT67URurHYmvkiNQdBevwbI1GZGbo1zAfDihkSl4rG8ygdysSIrWHfg9EfB87pHAy3dqMh98lcqJ/8MieUFjE5i5EvFTx6kaZfrHns+mC7hES13hJ2V+00hhXilMrzQ0cTNwF4jNgr4S3JXKgTqvcUZx3140qMUWT5Wth4U0y2mtScrNyx7HrUkOlYkPeStGwxKjg6eKd9iLMjbn34OJO8o9fxywHMpnlD2Y6yjwU9MwqTsp2amN1AY9EZCznhMl/iCASdNxrjXtmpzD1qYSQGjhpKPiMmiNx4s4q7VkmOk9TTqziEDxMsohrEMIEhbfAH8QQW/SXm3cZTEQUH0KmDaaZxtAFQpzLPMHoYZPU87ihitVg8VR8QgoN//q3IvYdO5EiblALCK12GB/6mdHvHZT77Whp5FW39rpEbmQPodPAJYXxR6bpGOZ26gXIG+FW0AS2mWwHvNDcJvkdzO6XN/x3z3w9QmYT2pe5lAmaTTRd4vdpDSKL585tk1BIMk/FBAk8HLm5EL159LudgMmixOt3xHQgfbB9PYyszFEvCaYuJUnrTRD+RMfnnr7/3xi1/Eyd3pz+a2Q/gNbJ6SL3uZU6cHTZorNgkuMF07d3V3N9DRIz5dImJzaFtY6GxVro2XUeQ5aSWm130IGkR6trqkmVMMrHV6Wy8PzWiw7Gc4pPVppuWm3yBvNF9zyER/QtOUR1G3FscUv0crx3RAgaMyGCiO8mpCFJGMa8worwmCQl5IOt4MmV0AWtit8+2mKQXb5XJ5G/jV6wCUuswcXZALotkbjLVqJuttC+Z5Q6rPw4a90oSvv8FUEsDBBQAAAAIAAAAIQD9vT2EFQMAAKwGAAAaAAAAc2tpbGxzL3JlcG8tc2NvdXQvU0tJTEwubWRtVbGSmzAQ7f0VO5MijSG9r8wkVYqbm6Q2OliMxkIiWul8/H3eCszhyVUGJD29t+/tuqqqgzcjnyjyFCppQ06HjqWNdko2+BP9NJKOWDVdFbybaWA3cRTqQ6Q0WKHnOQ3Bl/NiU4hzTS60JnE9zRSNvwr2MfXZt4ooZHxHrTMiLDQGSTjqsL2jFMpOK5KZEr+nJ5Ih3BRnitanBUhCji1T6MmQUu82aAKlAkw3mwZy1jP5PL6C7hPwJIlCOSsrEr/j0frLsoZPJtHIvkAt2E/UDtxe9Vgbxsk6UG4H4y96a3lTMTFDVTLxwqpimhVuoWBUQEwkeRyNVuaHaYe7GFMwTasFkOxSfajgxpe9E4eX7JWqQHaxZJXW4MqzXK1z5+V7c6TlVVljeYNoanrWioBdVm1C1lODN2nIKIfRTJPWoI9hpFBML9LVjTfjMtf0e/i43vMbR9jW2X5eSnAsNeC9sLT4aqjnG+iHLMXzwUSohRv14VBRs0J+29ICESuB5m/mODenfR5scnykK89kOzWptxpDWM4xahaRl/pA9LIlbg3K58ErZr+qUaNJoK4HyqWLnJLYYnDZ2AafjC1WjDX9gR02YT2qzwP7Hc0u4IwPaa1hSZlarGCwIqf1fa9BL/ybg4Zn0TKyiLlw8WojVu9rtvbFrmJ6HwpmNr1H+l56AXjloR4ZVnRrrpdjxm3nJ5OG5bwyRSGfP1quz85Rx731tmz+r72QvkBzyNQaj2KhXZrgurMkYFyaRTJmh86MhjubznpJ86Do3p+fSZJ5fA1OlSB42XHRoBx/fd7Lu95cXrcIPPS4nlOkjf3VhxsMxRaEHy32QPA+CXYEyzVgiD4eTSU8IeDq4sOIeNq2F4nYLmh63m1fJ8YyQQTDAnlEb5btpS7nd9x7OpXH+RPvrqVKK0x1RTEmjBTBmhbp+31yqd4Hasd1dOH7xb5pjssgLPbqRzSsB85XTHtnLmun70YxRkFnW/w9lJAY6z6G6b2JGL8Y1fqIaTTltLWP6TEKlrmhkUDWJeFfZhns0dzuekpxfVcf/gFQSwMEFAAAAAgAAAAhADr/8R3GBQAA4A0AACIAAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL2NoZWNrLnB5lVdRb+I4EH7nV/j80uQWUnZvTzpx4qFqWd1qu9uqcNoHiiI3ccAHODnbtCDKf7+ZsQOBdldXJIJjj2e+mflmbDjnl+WyUgvJspnQU5mzAl4sEzpnZqWZE2YqHUxXGyetY0/KzZibSTYTRktrWbEQU/snq4zSjgmWgTaROfYoTa4yl3DOWy21rEoDq2ZaCWNl/V7aelRt0szDqGfMfmRXD5UpM7C1n9nYVmt08fk6vf78bTBkffax22q1clmwp9LMLQCQUdxrMfhk4IjKBWAHsXFpE6kflSl1Al5FfPh9kH6/ufsyvL24HPB4crrlnd9TCTdLcmW0WMpIxKwoDbiqNAMlwjkTAaI246VR0xScfORtNp7E4/e9DxOmCibAaG4xchFPqs0P7PDzPXjYf4r09vtVvQ+t414EcNDh3cUPWKRl/NbglQX4Uf32T6l0hDIAOpkqx+P4sB0/RrqV8dpbjfcmxBDwbCGFjlBrUBFEyY51RlVRnCz8gJ/zODFyWT7KyshCrSO+1wdLQeNSALigjOhiIHM1dZILM10tpXa3tBLFDbFE5DmGn9Yj3ukQkyGWoFWsFq7P+U/Fkd9vEJ+/QbNaynLlYIPbVLIPlXLY+uFj1+8tKwckSsHZoIV+0rkunzRqs8FZU5YOhBpMb3laUN0CkXxKCk/TAlliZGKrhXKR4eP2vZ28Q4KBuYT2xMiYok6W5xjMaDBD6wdq1C2i3yjKBLpENObIImBTrooCfzsdrJVOqRcbfP1rcHHFJ22WPeV9xA8jUQFPZAphqVauPzIrCdGRaz+M9yb3XhUHdwIMQJzD9uBa8OKk0qgIOCRknsq1zFJOboGOItSg7zqpNKY0ZOdQY0YuSPI4BkhskDsqJe8SyMfNIkRLtZhcKyBXs072MT1CkIiqAheigm9B3a5H1n0qypXOeXyyVzulV3I/6czmWPuhsSbhlzAA/UojlJUn0ZbrTFbNdpzcbsL5MEB8TFhGQN/kwxaoFdFavG8JlLSFgiMkised95Nx77dud7Kr6whPE1AQbIOKhQRKe7Z2GL4c24x35w2RHYUtAtmbL5y9Y6AJVJy12ZnPV5DiRBkimFxYyaCK4332QS9x7chMgwaEkDM2uLu7uWNoBURCKVo4FX5SJLi8sr5M7AxOtDeVRqA9GZpjtoC0GMg6tIQeJxA+mjqKNbr8Wj3QBiqWNK02mchmMj1enTQywy+bl4UeuQ/BeAbVFGDERaYIoA+uLrXEAJMaf61otCvncZM1ak3UjZu17V7tUEHRi8R8KxlpYFP1KHXCbgXcVUCxKrVfaDOZTBM/PsdnuoaO0evREJsXsEZBPOCuswy3HkJUbeKkUYb+uPO9ZImdcQx3gQRju3LiYSExyUtMtb8/wejXABlX/iUSrOlZ4VOXPaE3qqQZeoazo989EqIEAbkeVS4N30eEIjdvNGyARHcLOq38qhc+6hR40XtBV9j7/1kJw4DTJ8+/+DCFptJQP/LLg3WljMxPc1fwEUa6x0afvw5u/h4xUTi4BGybmnc2Yd8E1OQTXUZrOgGDVsC1kOf5DxLlXQCPI/AzlBMQmN9r5HGYw2p+tVv5Cl8tl8JsQIeGEEQRlU+z8IyEG7CVeeSNEYPxGJbCZDM4h+8foBFbEHguBNRQ/kwNxj/ts67Za4SO7x8gd6g3jtusvmmE+zUA4LcXwyG1MkTu3czKXLJ+n3V97UE53LDRYDgassub6+vB5Whw9YMdv/sdBf8E92sWwbnl2PZYbBefdOmQrm3AtIM+sA0BChE867OzuG7u3uFD26IS5wycfA935sl+osN4PO5OjsIaUgfQQ8sTxvnzPiLEgyvUw0NTjg+9wht90SY+wbTSU9/BcGc4IEh63PtjEk6El7H6BaL7krmdDpxOAv4nbQ//T3aE1TJYa/AxAADOeZMo4w/BVxwedw7qeogJ/lMVLKU+laaYN2jbeHGGnu1B+Vt06z9QSwMEFAAAAAgAAAAhALZqsi0JCgAAfxgAACMAAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL2xvY2F0ZS5weY1Y7Y/aOBr/zl/hy5dLZiAD05frIXFVb9uTqtV2R22lSkfZbEgcyBHirO0MpdP53+/3PHYgDNPtIiCx/fh5f7ODIHif1hth11IY1epMiqKtM1uq2oi0zkVWpcZII7bKWKFllVqZC6uwJkpjWin+aKXei/Dfv1w/E5eizGVty6KUWiwVtpgoDoJgMCi3jdJWpHrVpNrIw9jY7nWb2nX3rkz3pg+gZm8GhVZbkamqkp5Dv/aTamsr9VDkskjbyuZlZgeDDz+/vUlev33/QczEXRCvShsMRRDfyvqWXrqnfyzbssrpJS8NA9Yql8lW5W0lDY2TpNlnabaWScJ4rPrCT7la8boprRw1abZJV9hwP/jw8dcbUDbShlBBShojhaZaQmyRWrGUYrkXGeYz1Va5yBW+UHWhtGBJ14Bbp7dSrNVOlIUoofMaui8huMUXVkn3Yluu1nawbWGfWuEL9RVC1QJYzJoRGwX7ggz+LFsaPxoBD7C1RtIvpz+Dv7JeiR0o7yQ43RHkbl1ma/qvMFFWFf4sxoR7sFetwE8DXNDDOcWyXYmi/EI/IKZ/iCVT25LweU4/Q39YNG3jXKGqICX/G5gdoLV/GAhVg2kNxEVagV0jq2KQVYbs7Ty084RGi6YFh1rCL6GRXJpMlw15i4Dpsk0F6wq7bySGaQ1LCUtw9Gdgptg0VWnDKBq8ff3m3UfYT8s4U9sGsoc6mL8a/TcdfU0W/mU8+meyuAiiwc2r948CL17OAba4/MaDy/Dl33gcffucX2LbYEAS7JTeGHiODKPpQOADn8jLHKFmgHOuTAwfLbWq4xU504dPb5JPv77/+cPNq5/eBNHi4ZZLt6dBQMV5qet0K8M0YrdKyYeAJLVWh4gouK3S5SpBXFIQzBfRfDK9XpCzpSCaG7J0GMTN/jt0gqsD89j/kNObT6+7fUSd9hIDRxxOXPqAIi/Tr2O+NGA/7Eb/U2UdEgwFHQVzFB2308c7DYEMeuM+i17hpUnI4CHymceBNzZfFWvZVGQKkDWyAa2rIIrhlFKHEYOSPjtY5y0AGYpJNB9NFn3CSIAVSABNqrM1HCL87dtVFLKrvfxGD4RaRHuJj4gClnDHxqbaes0TVBIc1442YQnYMN2imM1EkKm66Fa8tFZtZG1A94v10qrWkmMdLQMNsWXY6eOirHMEYn8HfSg+Zwx6og9vPVql5AMsnPnIkJWsQ8xH4l9icmorcBCnTQNxGOCw1pDoxFvT0WD+GkJLMXbgjdjwrtUxAGK8ncmRUoIkoH0Eek4dshH15ojf0b48kdDng4Bs0NEg/NcsX9OXOOrbHhS89rN1W29MooqQvNjZ2nFj9f7IFqdU1bAMBCbrTOXwj1nQ2mL0Ak4itVbazALvoGCIy0OdV/JUOl/HZ34RHp3mPVtZLWkxJR+hWhw6eAcgv2SyseINP5A0j5i9YN5pqrLm5OT2Oi3xnCdUoCLnzsd4TJq4LVEiQyqrQ+RpiarQcy7OD6gw7IbEG6qpTngmoS2GNz6IeKqJqFcImBoRy8BDEdLu//gm5rUshozuldnX2dnsT1Q7MHqYSnpxXgR3jtn7OyYQ0/x9cAYuWd4ut3pe0FzkCemlVrCg2++G0RmCpcr3wBB8rgOX61if8/4mMUIcAeXifPemZPoBV8Pgcc2cSCwkVdIAhjmXha3XxSdlyVPeyTuBjkgOWU1DZj465+rHbJzrnT7OVTw8G+ISST9wBNwiuTEU7OdAiEKRGe871Zkck6HLSqTbCPtdg+dy8NC783x6PcZnEZ2ENCPzQb1NYR/POweRpojyzW38Sq/aLfrgG17xEeHAYjQ9VG15PQxGI26fg0PnOusE+g64VQ2AqX+ZoRU8bpuM/2SbltzR1phB/rjwFFRjUf+RBnUK5v1GfiSbWu1qQtCFs1bKutzfNSo87Xp/eJ3wLjsnpDFPD8UFD4g43ueSQ1xSeDua3mSn9W40QsOwQNq16NrCE9sy1qNtG12SbDfU/FFP6xpPizQEtzgeQwwVAk6dgsoZ5Uzl2kHG1/nU0c5OsMQVTcjmqydDdzkyzUgbdztXO88LpwPuCgbVv5l4yhUj7GoSsZXW+zBDl4MO+FDpMl5FFwbvu++R++ZOEmgnOiI6+D089qOfd/HiIvodZnbEvTusdWqkL6hep8eCd4Js/jn4+yKc/4bH53px92L4Ynwf8eQBp8/lrqKdNhBo1Fzd8g0nPKtAF8yvRArd1C6tNiE5Ui/mO+j5dEH4coeLNnQrpMT8UGYPJzrSJU3mJ94TB72WgI9RnDuw88DMWQHhtPGgtzq0VcVZQwYS5ykLTRd6uVaeLJA+INRJ+3pQEyM+zZeuBe3AMaJn1zSQ2h5tTc/r4bG1JQk6fChMpvwqGR81ME/H4wQ57i/K4kze9UyPtTQnoeo9hPpSClwfTGex+051vUp/A46g6bLqLhbQBWxNLD7i1QVCP7T5lHwI/kdi2fUeKut5a17g3V8YHJoV7dM/Gfo71Y0PLszmUQ7I1UsRbFJxIZ6gWPkptsIFusXDDKGaT19weTngyYgd0+OLEB+XSYBDFXOwrooxWK/s5kXcNnSw8lDxRu5N6AFq4KdNhM3NpLcrauHabYj5FbyVFJEwahpRFBKsuBJdVjxjtLOtQ2gypWV+mhn6DAMram74Q11HRPsrkhXRH3q19+KO6YDMOB6fhDs5C5IV35R0zFIjuTXhg6i15AUegI6qbuf4LJxscR4hJbsQ3VfhlLAKJzBuWKM1y4s5oVlgPI6fkdrCB1PnDZKTBCdpQnpBbMFb4mvai/dLMcH7hQjHMd+ujeN/PMPQ2+eKDBiRf22RWv5AS/vkSMCs6Tpk5k6N3Tk1Pj2nehkdKNdlCrDuNNlNPDhcdAxPrnvKlxXdGKCc8TYOl161B5LoO1ie9JAcJskjn8fj3snMlzEQaToCJ4dPOoz3snQYkONc8W0eXhKjMx64uyC3ALa2TSUxeNj8Oz4uyL2enzJ37CV5OPyzrBH1QyI2UDH8/hZZS84+6ha5350BVUPh4i79uFK6DfMp92cx1vk2xszHC2Tt8aKfZrH4WE6FZ2Zr6ZOmQnrUD/shSpnO2MfO6DudkENbBB/B6J3LOU10j9DJJY78WeruYQuP1F8Ge+1061DRfrtUOPcGx4T7Yw2SNh4RsQiEuOPd0+fxpLjHCEjup3eM5X50BzQ0SYjuxZ07rjm6y31CjQA03rsoDotKpfZxxi6SvkmejhdHZjyuOeAWnA6+hP2podsV9ZQY/OLv0OVtWltuSUxfI04V7HxEFB4j8w5nl8WgH7mfVel2madiczsVo83tfLKI5tNni7+oqOCEp3cw/VTQFYG7GYaZO7O5Owlkhx36IcEMpjlzM3Q3yHTXVOqtWMqCmJZ5STdaMd9rwkWThHSfJHwvlSR0ZkqSwHHpDlCD/wNQSwMEFAAAAAgAAAAhADTTOGMtBgAA0w4AACEAAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL3Nob3cucHmNV21v2zYQ/u5fwfGTtNhKE2BA4dYYgjYDim5t0AzIANfTGImyucikRlJ5gev/vrsj9WInHSYklkTey8OHd7wT5/zKKu2Z30jmTGsLyUzFBNNiK0tWtbrwyugpK2rhHDOWvcOHbCv9xpTsQfkNq5WWTLfbW2ldxjmfTNS2MdYzYdeNsE727853j8Z1T7afdk9uMrn++OEqf//hyzVbsB3P1srzKePZvdT3+NDd4+22VXWJD6VyJKhNKfOtKdtaOnzP8+apEMVG5jnZ8eaR7nK9pnmnvJw1orgTa1DYT367+CP/9cOnS3R/dv5qMpmUsmIPxt45kJJJOp8wuAqhS1UKLx3ILY3LAI+yRmdr6RN+fXOZ33z+8vH66uLdJU9XxyonQacRfpOVyiLViUhZBewKpjQDI8J7mwAhANFYtc6BSlzwcpUuz+bnK6Zgj8Bp6XAHEp41T9/xw0978KB/jPTq5n2nh95RFwEMNsJy8QKPNI3/HXjlAH7Svf1tlE5QBgnGjUvTQR0vK31rg/XJ6H0MMRKuXA7OfWJlHW3AE1ANv1ltHqRNUhpF6uKwdU2tYEmnsMyzdDk7W4193BpTg7XMSWGLTWJ58ue30zRBJ+7nb3hTep2iLrpMMdDRdua8sD6SjFI57+fYYsF4YXSF47gBETv8K60wa1yCvEzZg9BelnEh3j4NrFD+mEbqKCl1YUpAsuCtr2avAY601li34FY2NRKUQhaxDTBYy0NyY/Iu4mRmpSgjS8GtxEmBUDEnkyAfBORjIRvPLukGwAfLkb5lIBMzHSM+6GbEOI1FRxvlKSFWE3pFNu4VpFiCaTlljQVqHkdBQTG3gRTGoENokI02p5EcVRwpHkWRwvBQGnZGQ0KS8JQlqP1LPK3ey2pK5i7cky6ejdIJBm/H4YnXP63AOKv4LoDd78hBhhu+58/EAUvQWMRNZt2Ksi5EhnGUHOcsZyeHoXF8UfABmi3k1bLMkGltiLSQpeSnlIWxwhub13AGrsDmMkwE8VX6omncqUw0EHllkpCfaX/sRErhjC3zYATicGwzndJapiEe0uceXtyiA+pfXnAIlihPxJ7AUcKDgzCJcQzY4lgMT1xNzL6tALKieQp0i1EfC1F2YdftVmp/RTMxaoNYJsoSj1maT/hshhsICwejoq39onP5HWnM3/8rbSUVKw0jkNk/RlHTeDju5aO3AiBHRbrld9o8aDTQJVoMqgVLUCvEGoQFB7boFKZRdIPhIClkJIZMsA7bo41nh6fbbAaVIE1hzKom6e78rySN+KJWcD3sX4PtA1QSbA+wiehaBsoEahoIHZxahs6WcK7y9OiICdtpDMb7qNx2nmlByPHo8FB1X36H+oMmpoN4Vsd1nPIUCNmaexkSO+G9F5iKNVDWTr7gYHVwYEHJC4d1LN2waShJj8gxwHkQ9R0hOYrzTmM5X6HdcsjlbgbXWhLPMDj0Qlh0cbA82LKsq90HiDPYZEzrA1p60FXoMyq0P4Jdseq4neidVsdOu9xzEOkjgtAuOulMu3EZwVO1QwRveI81j3jKYn1D1E42kOGnoxBBy5DjRNR/V9fBIYHrj7hxQ0E1frymhJemoHZQPoptg61jGqSm6DY9iP9o+TgBKv7JjMDF/nkXwP1g97CGVpcZ+90+QZPnNtDwytBHTDFTalNAuwW8h5YAE0k510rmYTe/ly1xjQ5sJXfyaVGL7W0pmJ2zxC5frWAFyzP6PV/FNXRYd7DxSdRP9yPciUux/x9wv0GsD9CSsLZh3rDzDsxtW0LBgF3tW+Y+CPIp/hF9XW2RujwsGriXEcASGtqBTnSnc5CPhY80QyU86ZzOoMN7xv9XDdV2B1738x3J72c7UN4ztkPH+6N4Ch8sBEPoteyADu5PwMlhUPXsBd35T2Sc1rOM5gDZauwJgmaw+BZpeNkkY1mWMcQLJnqNPdsaKwNjb1jrJMOuLsfkYhBjtFhWWbMN32C7A/BjGB1vixGaWc/q2RhuFH0L3z+vDrHegvO7UKogPi1lPpCB4bXnxKkd7+r5/PVq1Z/fpPGsanzVn3FinNVzhn0RfaSFUkaaEL/wWVmxPMeUyXNqvvMcqz182QW7ofRP/gVQSwMEFAAAAAgAAAAhACKAE9aXBQAAAQ0AACIAAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL3Rlc3RzLnB5jVdtb9s2EP7uX8FxA0Y2tpIUCFB4cIui7YeiRRs0A/rB8TRaomzOkiiQdB0jzX/f3ZHyW9KtBmyKb/fy3HN3Muf8o/GB6Tv4Ne2CBQ2zytTaM9WWabpui2Bs61lYqsAa3eKMKea3zdzWzDrW2HJda9aqRmec88HANJ11gSm36JTzup9b3z+53Zrf+sHg5sP76/zt+y83bMLuebYwgQ8Zz77p9hs+9GMa5mtTl/hQgtk4trbUeTTC4zzPu22hiqXOc5IT7B2NerGgfW+CHnWqWKkFXHgYDAalrtjGupWHVS3keMDgUwAGplSAApg1tT4D/cbZNlvoIPjN13f5189fPtxcv37zjsvZ6ZWzeKdTYZmVxiE4QklWAV6KmZaBEBWCEwAAmGSdWeSAFzo4ncnp5fj5jJmKKVBa+o0JS8GzbvsDPfx8ZzzcP7X0+uvb/h5qx7towF5GdBc/oJG28dsbbzyYL/rZP9a0As8goBgoKffX8eN0WLsofXAwPzQxAW58jgwTTtdJBjwB1PCb1XajnZC0itClZee72oBL5+DmpZyOLmeHOubW1iAt81q5YikcF399P5cClfhX33EAkku8iyolUpc464NyIYGMp3K+39vDT8ZSDJL9jQIokuXEcwdW9pzPXrvFGpPlmnaSK/FYpsoSg037go9GqAmsAqFqXYcJ5/9x2mkifQsrfsKfpaO2C0AjfRecAiPSRRryVWs3LQrwyQinNnBG4JWMsAVfOUSTQkurqIOdsakmwmhkSxQN/GgtFIwjyEYjoJeUsOZMJ/qR/y1kMi7dAr17qnTOoDfXyvsf1RKmPPqF1SaWFjk4JhlNf2Wvi0J3gfFutTiH6xiiIc0ymOHjmxq0ZI0OS5gzSsC5clHJHyyyhTwNSw0GgO++04WpTIFIhiyyUOuyjjzM/HoO7LoFTb+hfCSU2gBqXY3lg9jJwVyAItIVnndchSwKGshKggrbdFBuUdicA+KwpH2hOi2iOolruJciZwHGyWGlSlm9hnyFOrBPckhZTNchS6UHyIF1nR4xnJDOG1WvBEo8SOD+9HQ8Q3lllIUX+h0MZknhhMV93caCgYvlETOyvu70ZlUUV7i5M+a4eCSmVCepR1nH+vunKo5FUHm00KTatT7aQDzAqaNCtoOJBMuTQlYfHIcZjiIeJ9h2AYczXncQ83Mun3LncaH7X2OD2z4+iR5DRug2WaHbwpZQ0iZ8HarRC2Cdds46KAvJMgANcmgJwan1Y3GkB/IavIxHwCFVimMX9B0l1+ebdyj6J60vgJAoto6mIt+zyrSlqmuBGuWxDsQ+Xxq8EWnfl39iCoX8sB+coBuV9QTsZf0sztgbhqxYO6fJ5OlsyD7Z9vgUUrc2LTEX7Y95jStePBHRJiZ3owJ1oVv/TLwaK79ti1t/Jl9BmYeR2lJ+uzmTEDcUJR/JAQebp8O2t7fJFs6uO3H5+LqusZ2nk4hPH4rUH0kpbfSHUloTKD/iC2xlqgMSliJdkydYQTHqT4gYnDMmri4oXn2kde01u5DUiYdRqjxqFiTmtF1U/JPdv6T276Ke3Ufa/OIeMvbR2lUs5iiU2YqqegHUwzfc2F6GWEvg/bBYYe8xjakV1BatoKno331y8ul2E93z8OoqVno7qVUzLxWrxmxUTS9mqW8nY//cv04nU9GEvbFMULMBCPQ31eJR56Eac7kr476wTh+CRBmBJkzHL2Z7ePwSejywgRoPVTY6PR1fzbCFgC1MnN1jOkao2YhdPQAYTkuOiB/svGRXMTr8AIDeI8buwZYHnmSy0UtYIeUPJCfa0V8/RIN/WUPXgRSiIkbYQ1UfxheMO9V0EM/IO4bSo48Xs+nlLOoaj+93a89n8BsVHqwdqIV/IBXLc2JbziYT/EuAb2vwfyBiFl/dBv8CUEsDBBQAAAAIAAAAIQB1kWGSAAEAAJUBAAAdAAAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxVULtuwzAM3PUV7F7ZCBp08A907w8YjMzIRCXKFak8+vV1bGTIRN4Rx+NRMNMAoUw0omC6/1F1eUVpgEg5oz/6j8PJs/lfNH894uHTB3MTaai8GBcZ4Jtw8kXSfdsDgheOaKV28MUXEkABVm30DpWsVVGwmYBuGAxSCfjYss5KMQjYlFbBBAhnvsGSUDrHolZb2N3eWEJqq0/X9UsteTHtn6d3eXKRhCoajaGIkdijnjm+CndOe8W8JJbY3TEnZ6UkHRyAh9pkFa4PkGnHa8bxzIk2pIQ1zKNy5oR1fMTe+Ei2gVGI43wqVV9pbadYcZmd/nB6Wu19X2kpXkNp5v4BUEsBAhQDFAAAAAgAAAAhAPqilUocAQAA2gEAAAoAAAAAAAAAAAAAAKQBAAAAAGFnZW50LnlhbWxQSwECFAMUAAAACAAAACEAEq4KRGsAAACCAAAAFQAAAAAAAAAAAAAApAFEAQAAY29uZmlncy9zYW1wbGluZy55YW1sUEsBAhQDFAAAAAgAAAAhAH7UgdxNAAAAYQAAABAAAAAAAAAAAAAAAKQB4gEAAGV2YWxfY29uZmlnLnlhbWxQSwECFAMUAAAACAAAACEA0mljqkkDAACpBQAAEwAAAAAAAAAAAAAApAFdAgAAcHJvbXB0cy9hbmFseXplci5tZFBLAQIUAxQAAAAIAAAAIQC2lmKg3AUAAOcKAAARAAAAAAAAAAAAAACkAdcFAABwcm9tcHRzL3N5c3RlbS5tZFBLAQIUAxQAAAAIAAAAIQD9vT2EFQMAAKwGAAAaAAAAAAAAAAAAAACkAeILAABza2lsbHMvcmVwby1zY291dC9TS0lMTC5tZFBLAQIUAxQAAAAIAAAAIQA6//EdxgUAAOANAAAiAAAAAAAAAAAAAACkAS8PAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL2NoZWNrLnB5UEsBAhQDFAAAAAgAAAAhALZqsi0JCgAAfxgAACMAAAAAAAAAAAAAAKQBNRUAAHNraWxscy9yZXBvLXNjb3V0L3NjcmlwdHMvbG9jYXRlLnB5UEsBAhQDFAAAAAgAAAAhADTTOGMtBgAA0w4AACEAAAAAAAAAAAAAAKQBfx8AAHNraWxscy9yZXBvLXNjb3V0L3NjcmlwdHMvc2hvdy5weVBLAQIUAxQAAAAIAAAAIQAigBPWlwUAAAENAAAiAAAAAAAAAAAAAACkAeslAABza2lsbHMvcmVwby1zY291dC9zY3JpcHRzL3Rlc3RzLnB5UEsBAhQDFAAAAAgAAAAhAHWRYZIAAQAAlQEAAB0AAAAAAAAAAAAAAKQBwisAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sUEsFBgAAAAALAAsADAMAAP0sAAAAAA==', 'anchor': 'UEsDBBQAAAAIAAAAIQAv4uizDwEAALwBAAAKAAAAYWdlbnQueWFtbE1QQU7EMAy89xXmThetWHHojZdU3sZNLRKnxA6l+3qSVgJycDSjGc3YgpEG0I3GKTnKXawzDOApRuxv/ev13rP1n2j9dsPrWz9Z50inzKtxkgHeiyVJMRUFTbNtmAlIPAtRBlvQYOZvUsi0JmVLeQdWLZXZ2BaILBwxPMMXZZ6ZHKxo00J66VjUcpnOmCeWKRRHsOYUV9MX3dUoXqLrPAlltLaAGIm1f2b/z3MS1YNxDSz+smMMnaUUdOgAeshFqqtuLO7EhG6cOdCByLH9oS1zzfqFnmxUQyt6QC33WNXHEgeBvlVqWS2qvbNNkyxD04+HRF/a/UcUDPuD8lnxNOgHr6OWWi/zA8971MtQ9wNQSwMEFAAAAAgAAAAhABKuCkRrAAAAggAAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWw9jEEOgzAMBO95hV9Q0apUTT5jpWCSKGBHYEt9fgOH3kajnVXaGu1RbacAw+3hVBq2E/14cQ3wHNwWvyimzRRVKvER4H33fZ0L18IJJ+GlpOAA/upjcyI9c//qvvC02kyoWSxl7Q9LXA9yP1BLAwQUAAAACAAAACEAYCIzrKwCAABXBAAAEwAAAHByb21wdHMvYW5hbHl6ZXIubWRVU8tu2zAQvOsrFs0lAWy3TdGLERgwHKcI4NpF7ALNyaLFtUSUIg0uFdlFP75DKn0dJPCxj5nZ4bPvSAWmsvKa98ope/nBoRyRosBKj72zF0p35NSLqVU03pGcuDLKGokTekYBxy8cqPXaHC90NJZlUnwyL+xIOTIiHY9w7DTxWVURBfuG0dNEajuJdGDcnllPiuLqinbeWynGVIbO7Svftsrpko4+0NNyfj/erFfP9HoslOBNqawDn2gcHHCXVtJfWNM472u0sb6mMSK2ZSosrELV7MW0xqqwT+yGBpV3FZ+iUGx4wE2apQrmwEK9iY3vInRojauzJqlYzTFX2Ds2dXPwQUqKnnplv1OlrOUgUEEPa5b/UqQ71EGdmpwhzNT4HsIfuUe400aryCSX9gBFEjjHVczKYDL7pHOZUVFEZ5A0jikoVwMs6iH+aEKbNf3MwK6L9xNanmPADMhodtEcTYJ3DL79S3lKx85Vac5vK6tEEmGWEXEIkAhLUXXap/50UrHB2p9SPKZ+O6FtVjfryQoL7zB9VHc4shYEU6ekBlWNMo464LB0gY1Cjk/XmcpgkgM3MJ7vAmk4KiRyGW/fqPjPnPgMT0Yg+DChxcCcDpfs4TStFAjanbJ5cBNaZ8vWHegM3VzXHiDG4MG5kx7X4NCiyzW+1sOotx/fUe+DBmPnY5MKsxW+KVabxXz3uFlP6S4pMpveSVQhzsZ37PSMru9+S0rJZUnV2U3xtNnsaDH/ul0iDSqlu9gnI7jIMKLMiofHb/RlNU91kzcDww9QDSOeFU/L1Xy3vE+5YBcGScVECIRC+RXiYfIf+oIxviaPUsQbh55vZsVuud1tUYXPeM85mEF1yI9JZD5zqIwwdkayfLNisVk/PN4v1wtgb+A++glnaNO1WGDGxS9QSwMEFAAAAAgAAAAhAIqHvrRjBQAA1QkAABEAAABwcm9tcHRzL3N5c3RlbS5tZIVW227bRhB911cMEhRJBZFuk6YojDSA6jqJUdd2badpn8wVOZQWJneZvVhiUPSxH9BP7Jf0zC4Vx33piyAud2fPnDlzhr/bSMoxKUMqBmtsb6Mnz0ZbRxdj2FhDbNbaMDvaWnerzZq08brBGfLKNCu744bqDde3NgayLdYdq47swKbwNrqasTBYr4N1I6lABxLID6rmcvbGqu4Q773t7pjChkl7H/Fr0kP0uLdn79WaaavDJq36XnUd+0C1dY7rQIMK9WYh7wzVeEeVj6teh5v0oipns8eP6a1yDbmIg7OCzvgOkbnRYUGqaQjpNtxxAAYE9guqamta+V8OY4XHYUwP2mg8HZ3IAWRwq9bCSKsRtaS3ummAIEXItA5Dp0GPQiRHI8huEcBvSiD4iXmgIa46XdPy4sTTCuG2grG2PXDrVYf8DQL7z3jhHSLWOnRg0t96aoFDgX5l1ixRr2onKWdEtLYULB2EfiBrurGkpUFNBXHHbRCS72tBK8bFODQol+oIuC4zK4GvAYHNnXbW9GxwFhsdF6uou3BIjSVjAwUUGBdCIEGqkAkCMzi/7LZq9FP+tBpTnQTJf0pFS7xx3MYOgHwouEWKAed2eFbg1dg9KCnqe8BvO7udfV3SfP7ONOxwt2nm80PCn5A1BdYgExTiDlVSdYhKom/UnZYcgVhy9dy1Qok1LNUNWyuNENjUksEziX9qgVp/ZESfEVFBR5Km3AC5NHyjjOrGj+wqxLTdvWCRTTcVMPBOsgHKkk4ClB+iM55Oz4+W1yfnZ3RAl+fn13S0fHd1jIfXJ7/RxenyrKRf2el2JA0K6k7pXjhEozXCIfrUS5ZIjUAqirhi0MZJ4NhQZrTHu+Bki2gfkaDVoFvNbkHsnL1vNPCXBERG9QDvbP+ZAuWlZ+VEZELThvtDqtZocSqcoUcv7+O+ekRFoU3dxYa/n6ORqKQ/aAPQVDz/qppQXcqzyDNdkuFLq8BzGnpaSY43gqbKfKq0hZwoXupUeWwrDD25ul5eXi+Oz34cnoC20+Pqy3L2XKp2yYOzTaylbLR1Ooh39RAibIR87fQQ9m1y4GSvIA0bOJXf2G3uvlVcp8xdNChBRiKeIAb54CCy+kYufa13cp0UgCYXzD2Zj8p6zqqkI4v7Kts1Nz44VKsi1GcFB+jvuZetC5pnMnMD59JrI3RjszXzRS4Oggz4NyaiChP7FcwHzYoeEpdKxvPgOqSJFlvBvqPRHyKXdA6GO7vWkPtkLjRMflkSUkuYnEXL1woevcjdL9Y89kOwfUaiOm8Jp5tuasMGcWrl+XNHEzcBeLTYC+Ety1yoE6r3FBc9DeONGKPI8qWw8aqabDWrOVu5Yzn1oCDTWJB1ubTu0CoYHbzTPqTeEbc+fHiTrNHL9OYAZrN5RcWOig8VPTUKnbKdiljcisa+FdhXycQ+wV5L1VGV6KvERFpodNtiePTqFjxFdGcSvRaLaaDhSSWZbo9UeqXN/w21HyI4ldC+1oPUezaZUoXlmz2ETNGf32VbkmDQwXsJPI0XzP9nL74Q1892JI2te/4Ewgc7pNljRTEpJcwW6Efp9SZ1j4jin7/+3tuU/M2cfJp1+DwYItwrqe2QBj2IKpyNa1AppgBuoKW9l5j7qSvUl9PIHgAA/bex0EGn3HpqK7nlpJXvmNRxrdId8trqmkUUhZjINAnuPTL1M8vMmowlf1e4qQvIGz0MHDLRv2Bm6DBiSjtc9XMasknwEYKIJvWizACQMkqrhhHpYdSGfA+cNPlwQRdoRHb727BRkgjpGypb2l16i11Aal1J1zbiLovL3GQhSTdbKV+2hh12f4gaX1EkfP8LUEsDBBQAAAAIAAAAIQDScUX58gAAAHcBAAAdAAAAc3ViX2FnZW50cy9jb2RlX2FuYWx5emVyLnlhbWxVULtuwzAM3PUV7F45CBp08A907w8YtMzIBCTKFak8+vVVHHTIRNwRd8ejYKYRQlloQsF0/6XqckdphEg5oz/5j+Ps2fwPmr+e8Pjpg7mFNFTejIuM8E24+CLpvvuA4IUjWqkDfPGFBFCAVRu9QyVrVRRsJaAbBoNUAj5c+q4Ug4BNqQsWQDjzDbaEMjgWtdrCM+2NJaTWc4bhsNWSN9PD/+lDXlwkoYpGUyhiJPaYZ46vwienB8W8JZY43DEnZ6UkHR2Ah9qkC/sDZHni3nE6c6IdKWEN66ScOWGdHrV3PpLtYBLiuM6l6iutbY4Vt9X9AVBLAQIUAxQAAAAIAAAAIQAv4uizDwEAALwBAAAKAAAAAAAAAAAAAACkAQAAAABhZ2VudC55YW1sUEsBAhQDFAAAAAgAAAAhABKuCkRrAAAAggAAABUAAAAAAAAAAAAAAKQBNwEAAGNvbmZpZ3Mvc2FtcGxpbmcueWFtbFBLAQIUAxQAAAAIAAAAIQBgIjOsrAIAAFcEAAATAAAAAAAAAAAAAACkAdUBAABwcm9tcHRzL2FuYWx5emVyLm1kUEsBAhQDFAAAAAgAAAAhAIqHvrRjBQAA1QkAABEAAAAAAAAAAAAAAKQBsgQAAHByb21wdHMvc3lzdGVtLm1kUEsBAhQDFAAAAAgAAAAhANJxRfnyAAAAdwEAAB0AAAAAAAAAAAAAAKQBRAoAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sUEsFBgAAAAAFAAUARgEAAHELAAAAAA==', 'fallback': 'UEsDBBQAAAAIAAAAIQAr2F6FIQEAAO0BAAAKAAAAYWdlbnQueWFtbFVQSW7DMAy8+xXsvU4QNOjBD+g3BFpibCJaXJJu6ry+sh00rQ4ChhwOh5MxUQd9RH91Hs35EkiaVP/YwUApYXtu3059y9Z+orW3M57eW29NIPXCk3HJHXzwNymUTCA0FWUrsgCrzgQ3thEQNGGMr/BFwhemAFpm8QQTmh8PDWc1mf0u9sLZxznUppQ0mR51UaN0SKGxUqJ2DUALMudqtvrLYceEwV040oYosD3RTdjoCQcyp4Y26wZ17lNlb1Z++2sOLhMPY1/kwSMUPzrlxBFlY/ynV6FBcNpFcKBsbjW8+l2fL/nCw7pn7NalbqPocZvFjHG5kxwWTPExoFeeqmi9UfiOezg1JmrqHAnWk6qkrWt26T/R7YUaHaYpch522R9QSwMEFAAAAAgAAAAhANSUvOBeAAAAagAAABUAAABjb25maWdzL3NhbXBsaW5nLnlhbWwdylEKhDAMRdH/rKIrEBnGAbuZUJzYlmpS9AVcvuP8HS4Xsnc5EvyQGMbhRbDO/eE8/d1ieI+0p4vN0R0Ma6LnU+cPoVRtVTMvpmvNkUKoumz+FUYxzwW/c03bKXQDUEsDBBQAAAAIAAAAIQDH8proTQAAAF8AAAAQAAAAZXZhbF9jb25maWcueWFtbEstS8wpTSzJzM+z4lJQKMnMTc0vLYkvTk3Oz0sptlIwNDIACucmVsSX5OfnxCcn5uQARU3ggkD18bmZeaUlqUBhU5hoaVEeSK+BARcAUEsDBBQAAAAIAAAAIQDfwDyr4AMAAOsGAAATAAAAcHJvbXB0cy9hbmFseXplci5tZG1UTW/rNhC861cs8g5JCltN076icA+FkTiAgTR+SFyg72TR1FoiIpEqScVWP/57ZynHToEekjAid3d2dma/up6UZyq0K3mjrGqGP9kXE1LkWZVTZ5uB5I6sejOVis7TDj+xZvoyxNpZvOtcMLgYSEX6du/8a+iU5py+IrnlN/bUutLsBtqZhkNOawRLTk+BbRloEBAUaucjsv3Rc4g/pwq7vmnIhNAzfkt2+YgQcjs5Bny2IfpeR+MsEj8Y3PFB6QjU+5rRmESMGdo+RNoiqTlwOZELS8qGPfs8yz59orVzTcimVPjebrRrW2XLIjX7kYn0OZD8N6OiMpEqEEBTSxeRD/GCplO6/Cbvhkv6m2oE0vT2BnwW4zNv5Ry4lIjLl/X8eT1ZPN13l/SwfFzIXRPSayRuXIVsznJjLNP0s6ROr3JgrAqAL6mInrlII7QuJjpU03CZ06qPXR+FN40/0ZGJAb17kPB5cnNzQ7pWHkyxDxMKjlSzV0OgznSMsSivwS9I8q6vaiqkkyJP7OC0kUEWtDexpmiqGlgFole2QtAVBtU61FFbh9I/3aTbcJ3TMibiSGnNHeB0KtZBQBsI7INypE7FcZNEaRkVts6HY8eni9BvK6+6uhiTvlq3pzBYDczW9YEuNKgIF8SloJJYizblBYRl3zXzlATaqRCokJebOHQMipc7aDLlB3muIfbeJaqi66gPxlZgdFTOrwwjlOe+fxwjUvnr7LucFocoVBPatNHsDDinnXftWZ2zE6QEVDeCx6qWUTFVJhyDquR/153edRhhy5jh+1sZzEgrOqisgy46mGh6tBVFbrtGRfgCYqXQ6xqMQAqsXxsTYsiz25xe0vhPPk89hY41kOsPPYxqgutc04D6eHS1aE3OSRJnE265xgZxvafSgHEZSaJgXx99PbqUDygUxcuormjHKvbIcMQvDaLtc1bduCB98QHoZSbvAdC9QavcAi3cpv4TZXn/AU9aDF3TY6A2z74/rpG0at6zRqmRuI0CV7s3GeHxXEKwP+R05ywIaWk7pIWRwo6UvKusQg9hJMb27RY5RgHN0xo6C+gW9oQbSgwUrq4lFTeBr7PH1d18vVw9zdKMZzC7j1NZiFcn+YC2pJ7r7Hm1WtPd/LeXxQwOYbmJeyc7N7LVHLKH5e/05XGObCNQqz20JIsBPp6ADd30qQ/xBeQMppPOSlydR5Y9Lx7n68U9iuAbrNTAwnjIfOIgIOiUFjAs4GTrxcv6ZfY/HF+F66P2ZHmBvuxu9fSwvF883aGTGtsAqxUoTN/iAOmN6/uE56/Ouy0mvyk5aG+SW375J/sXUEsDBBQAAAAIAAAAIQDTmV4SwAgAAI0RAAARAAAAcHJvbXB0cy9zeXN0ZW0ubWR1WNtuGzkSffdXFDxAbC8kWc5sBgt7dgBP1sEEG8RBrNnLk5rqZqu56iY7vFjSYD9+TxUp+TKzL0qkZpNVp845VfS/XSLlNf3cq3pD71WckLKkUnTWDS4FCq6NW16h7dpYrT21ZmfsmpzVZEJI+LQUO01f9rFzlrweXTDR+T2pSJdb5zdhVLWe0We3cs2etqbvcUjYYq9vSYdonA0z+icW4lnsaO+Sp+hcHyjZaLCYz8RhfNKIQPWED7RUK+xUhbQaTFyOKtbd+UU1Ozn57jv6xW15H8lt7VWjm5Mp/WKaBq9FnBnIJ+S5VsaGmE8M+Kg11Z2yay1n4VyvQ4efdL1xKd5InlGFDY0qBCxytt+TaWnc86akdwY7z2d0i9i8blNPK/w+1W3rfJQsVlphCTDWwxj3JFHPENvHlqIZNLmcuuSWg8TBki8xlIx89JqhL3mRQqwR3wGwfgSmnj8jggdUKZhVr/lgPmPBMbam12FCVQ55Zqyp8K12tpXv476Sh6N3/9F1nEU39PxL0DGNs7pdVwi+oSq6nbwrCAMlHZEassxwI8wJACXdmCghd3ogK+F1uh8R6wdgwUkdUHeNzpV7wPYrt6NW1TEg6Pu27UG8CTV61Bb1qw2AV73XCmTi8gEp3RyyLz8A2HqjUEfO+/ZRmV4BiGtaG4C5BkcnAMI2CFLjQ203E64h+Ps9NrpfkHp6pfLI+dybkV+7KNmjBNWMHrTydZdZW2Fr2ZmmtuJTKxRvWbthwBsVV3FMXCaq8U90xEz58PHrw4LeTebzOfPOI2XtAU7FyS25UhWwjcnbwGIaHOp39W5ODEhgqlzNX70rqHstuiKrvIcOeDV5YXWO67g56rr1JupylKTGJSvfhd3MOkbVNPqFmt9jm4j61Z4pnGnFormMw1gQeQ7ApJT/9zsxMTtmgI16F/GqbRA0gAoDF/L8+7cb4LXRNlzM6A577Kl3bEAZ0ZUHwYKIve5d0KwfkB9hQbmxA2yNCVBjE7LM+WBBaaNRq7wJzuogUEFnreMSFIopwEwyJK8cRigPZyiEZeNqe7c9uZrRryCox9sWfHyIjA9zXO9GSImlit34B9QqKfaGTj0aDgq4sZ+y+LcOnAQUtka56MENxWaDHDumvp8e6ttooG9GMdBrChszZqvqTRCPwWFwmV7CALSTnLKoLq04yuzKkSNESODLwNaAj8DSkR0eVZ+YN29n9MnBlMxvXHsxXtbsUlnV73/TntnC231kay3xAdTUN0hTkhOIj7llinAsGw0ZQ9fRtAbgAVk3yANJ+4bN7aD2Dl7HT2CsfWk+whkONHeUkPf9dP/+dvHx/vOEvt5Dze9vf324m0Bu/6Ivn24/y/rF3cPiAQAzajgic527yvHojEepIOPDHJd322RrRp2plvsFMIGDGj+Irkt3W+2J484GCGoW3Z4fpKxWoB/9cNDziFeeiRNsR1MQa0cfDMkLP7beibdm52FGB9231y/sh06f8Dyl6ZTO/gRfP6P/wnxVQ9O3cxgUjO6rhss3iYvBieejwCA1oh2zL2SuCMcyrgo7eN24WmDg7sQvZcwC0KIi9+sTIhAEif4khnDp+SgEQT/+SGd39x/OKprNYHT4b5WbeVX899X6tz+9uULgEXbMmUnof5b2MaOfc8PRCkiwbQEW0dz/UROyVNLr8wiSCyeJbLs9tIsCPXe/nDACc30DR2CbqZDfaCBk4QMIc+QqvzLBwXWfpN6GOxWCYZLICdiMi82Vnto0rFDrEROC2elwk3VpYpHICuuSNdDQU5v0zkXMBCngFMTdgIharFA3yKFWQb/m7UTOlZ3HtOpNTcGsLXwNzZpW6I6Y6hquF2yNhwTuZtgRtpR9t3bOIxXFvsWoZP9B04QkGd0u4WB+gNMta5z7mmtbVgIMPrETUgWqhGXw9SVDh2ku8LhSev4rtYF7WRAmHruLQtfYZt29FGixGMTWaskJC3XDAxlc+d2M/qG9afcHJhRWgh/wW1Tnef+AhKVlCaGLWjlbzetXWjCXkZGATmPqAsQIPvDxvQoxs/3A3+mAUYL73shhA93uMrrLzLaGKf3mDb1YK5vLUHrJn8sdL5p+o+mOpiNZd63s3jiaOpkQEfZf5wdZxatZ79Yikpsnjbx7/rDi4D6Y0nmkRR6O+93g0tMpcyd7Ro7ouXNUxxETwl8xCmWKrFhrCtVxfcEqJNgH2vUOrYgRbRGbMC9Zr3shFYCXhlx0eD6ABrLU7ISkPDHE3K3NMEIY4UKQt64M7HAI8Ha4ocbJj9ElGIEcL+Mk26cs9MXnRI2IB+zmeHACZmVTpCP29eIOkDvayQ9QhgwAMEzkLXDlAQEwiWLLmMAPGtO2R5/FqFaxyw7ukX0GNef8IG02JB7dxT5eDFV/dKNhyUgezLWn20G5BCE5FgZXkqkuwx7SFK8rpCseV8zvkaUBE8vTy4K38wnDG4afXN1yrZIOswxmwCDsl4xodZ29FsVJPevee6OfdeSCnmsLE+o+35LOcYdEpI43v5q/HlkvJlhusC0/50GqZHsYCDl4Ye3x9nWc02QCsdqsu5Xz4VCGwwPAiLvI2JVBdmMxVoa9rTu0UL7ansod61QsNA87FpMjrzg2+OOliu973B3Wehn3I8/+IJci2T+XRGPWlvk7upFSyDYiu7KBHEVWHS57x46pDnREtzJ8ty3sGzQ8orkh9DjuNxjktBhUmRaydnF9YcmfPSxuvy4md5//Np5hzPl0V8lFiFejGs+GCj7979wTcvcoHigsxgwhIselgJ2E/z1cZ47XRdjS0QTEAUulwZB1boIh0+oL6FwG+1UCajJaBNRcNyXyl9Pjgfqvpm/wxpeu8Jfn92IZ+Pjyfrh0Cqn3AF3ue2Zgd8FwrTOVMJjx7bzJw4r8CYGHM65Bp/p2q3ARd8gGYw9LKh+o6FtSngNwuVx/dD/XA/8FoVQdc37veE645j5pQp5v82RS+sEkT+jyQLyCM8l6n538D1BLAwQUAAAACAAAACEAe5YXiigBAADQAQAAHQAAAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sVVBBUsMwDLznFeLehBYCA+EFXPlARnUUxxNHMrZMW16Pk5YZuHi8u5rVrhgX6sDIQD0y+ss3xWopyHdgaVmwbuvHw7F2Wn+i1qcWD8+10WqgZKIL6oQ7+CAcamF/2XyA8ctZVIkNvCugj0W+wMxySqATgUsp0xtE0hz5StEZjcLozuDF4Oq6gyiiYDAn2m1K8MiAPIDxkigpaHlSUzlOGrO5RrlzbHwuIZrmPkRZgqb7317NMlQq4lNXAdQQM/dGSkUerrjE7EfnaUOWtN+uwuTsdJSY/tMpH23EMFWWmCIqFZqVeJV5dHbdobSEVcuxnHjfHFZKQh9W8Pp0Q3MH7b78Fzz3kjVk7VVm4tTBw759Wacmx7Nj+8cZ4Naz10mynbRMj+gTVT9QSwECFAMUAAAACAAAACEAK9hehSEBAADtAQAACgAAAAAAAAAAAAAApAEAAAAAYWdlbnQueWFtbFBLAQIUAxQAAAAIAAAAIQDUlLzgXgAAAGoAAAAVAAAAAAAAAAAAAACkAUkBAABjb25maWdzL3NhbXBsaW5nLnlhbWxQSwECFAMUAAAACAAAACEAx/Ka6E0AAABfAAAAEAAAAAAAAAAAAAAApAHaAQAAZXZhbF9jb25maWcueWFtbFBLAQIUAxQAAAAIAAAAIQDfwDyr4AMAAOsGAAATAAAAAAAAAAAAAACkAVUCAABwcm9tcHRzL2FuYWx5emVyLm1kUEsBAhQDFAAAAAgAAAAhANOZXhLACAAAjREAABEAAAAAAAAAAAAAAKQBZgYAAHByb21wdHMvc3lzdGVtLm1kUEsBAhQDFAAAAAgAAAAhAHuWF4ooAQAA0AEAAB0AAAAAAAAAAAAAAKQBVQ8AAHN1Yl9hZ2VudHMvY29kZV9hbmFseXplci55YW1sUEsFBgAAAAAGAAYAhAEAALgQAAAAAA=='}
EXPECTED_SHA256 = {'v11': '78eef3ef5943969e82a9e220ca70f5a21c8be49acdacafe8e03786ac411de126', 'anchor': 'f3534769cd8c7761b8ae83722b0f2fd381c0349587a10e84bd63e0554a4a7aa4', 'fallback': '18b20200ba4a42f99ecaa0c889664ead4b78848835317bc114f4e4095f759916'}

def sha(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def unpack(data: bytes) -> dict[str, str]:
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = archive.namelist()
        assert archive.testzip() is None and len(names) == len(set(names))
        return {name: archive.read(name).decode('utf-8') for name in names}

archive_bytes = {name: base64.b64decode(blob) for name, blob in ARCHIVE_B64.items()}
for name, data in archive_bytes.items():
    assert sha(data) == EXPECTED_SHA256[name], name
    unpack(data)
print('🐈‍⬛ ANCHOR VALIDATED: public 0.12 | SHA256', EXPECTED_SHA256['anchor'][:16])
print('🛡️ fallback validated: public 0.10 twice')

## 2 · Build one bounded challenger

The v12 delta touches only the coder prompt, analyzer prompt and per-task time limit. The agent still uses the same Gemma model, sampling, tools, skills and ADK structure. The state card is plain model text, not a file in the repository patch. The second diagnosis is requested only if the first attempted repair fails its reproducer or closest test. The five-minute cap leaves a theoretical two-hour margin under a 12-hour evaluation across 120 tasks, before setup and verification overhead; actual total runtime must still be measured.

In [ ]:
parent = unpack(archive_bytes['v11'])
anchor = unpack(archive_bytes['anchor'])
fallback = unpack(archive_bytes['fallback'])
challenger = dict(parent)

state_card = """
## Working state
After localization and after each verification, write a plain-text state card of at most 100 words in your next response: required behavior; confirmed source location; current edit; last test outcome; next action. Keep facts and hypotheses distinct. Do not copy long tool output into it. If earlier output leaves context, use this card and reread source before editing.
If the first edited patch still fails the minimal reproducer or closest existing test, call `code_analyzer` once more with the state card, current diff and a short failing output. Ask for one alternative root cause. Confirm the returned location in source before the next edit. Make at most two repair attempts; reserve time for syntax checks and `submit_patch`.
""".strip()
analyzer_revision = """
## Failed-patch review
If the request includes an attempted patch and failing test output, reassess the earlier diagnosis. Identify one alternative root cause from the supplied evidence and confirm it against source. Keep the normal answer format; mention the failed assumption in ROOT CAUSE. Do not edit or run a broad test suite.
""".strip()

assert '## Budget discipline' in challenger['prompts/system.md']
assert '## Answer format' in challenger['prompts/analyzer.md']
challenger['prompts/system.md'] = challenger['prompts/system.md'].replace(
    '## Budget discipline', state_card + '\n\n## Budget discipline', 1)
challenger['prompts/analyzer.md'] = challenger['prompts/analyzer.md'].replace(
    '## Answer format', analyzer_revision + '\n\n## Answer format', 1)
budget = yaml.safe_load(challenger['eval_config.yaml'])
assert budget['evaluation']['max_time_minutes'] == 10
budget['evaluation']['max_time_minutes'] = 5
challenger['eval_config.yaml'] = yaml.safe_dump(budget, sort_keys=False)

changed = sorted(name for name in challenger if challenger[name] != parent.get(name))
assert changed == ['eval_config.yaml', 'prompts/analyzer.md', 'prompts/system.md'], changed
assert set(challenger) == set(parent)
print('🧪 CHALLENGER 01:', ', '.join(changed))
print('⏱️ RUNTIME / BUDGET:', budget['evaluation'])

## 3 · Validate and package

Checks cover archive integrity, allowed members, includes, model identity, agent roles, tool names, skills, prompt placeholders and the runtime guard. A failed challenger check selects the exact scored anchor. The baseline is retained as a second recovery archive. Compilation on Kaggle is recorded separately from these static checks; a clean notebook run does not prove a higher SWE score.

In [ ]:
class IncludeLoader(yaml.SafeLoader):
    pass

IncludeLoader.add_constructor('!include', lambda loader, node: loader.construct_scalar(node))
MODEL = 'gemma-4-31b-it-qat-w4a16-ct'
TOOLS = {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch',
         'search_similar_code', 'get_code_neighbors', 'get_code_subgraph'}
WRITERS = {'edit_file', 'write_file', 'submit_patch'}
ALLOWED = {'.yaml', '.yml', '.md', '.txt', '.py', '.json', '.safetensors'}

def resolve(name: str, ref: str) -> str:
    return posixpath.normpath(posixpath.join(posixpath.dirname(name), ref))

def validate(files: dict[str, str]) -> None:
    assert 'agent.yaml' in files
    assert sum(len(content.encode()) for content in files.values()) < 3 * 1024**3
    for name, content in files.items():
        assert name and not name.startswith(('/', '\\')) and '..' not in name.split('/')
        assert ':' not in name and '\\' not in name and Path(name).suffix in ALLOWED
        assert isinstance(content, str) and content.strip()
        if name.endswith('.py'):
            compile(content, name, 'exec')
        if name.endswith(('.yaml', '.yml')):
            yaml.load(content, Loader=IncludeLoader)
            for inc in re.findall(r'!include\s+([^\s]+)', content):
                assert resolve(name, inc) in files, (name, inc)
    visited = set()
    def check_agent(name: str, root: bool) -> None:
        assert name not in visited
        visited.add(name)
        cfg = yaml.load(files[name], Loader=IncludeLoader)
        assert cfg['model'] == MODEL
        instruction = cfg['instruction']
        prompt = files.get(resolve(name, instruction), instruction)
        for match in re.finditer(r'(?<![\$\{\\]){+[^{}]*}+', prompt):
            key = match.group().strip('{}').strip().removesuffix('?')
            assert not key.isidentifier() or key == 'problem_description', (name, key)
        declared = {tool for tool in cfg.get('tools', []) if isinstance(tool, str)}
        assert declared <= TOOLS
        assert ('submit_patch' in declared) if root else not (declared & WRITERS)
        for skill in cfg.get('skills') or []:
            manifest = files[f'{skill}/SKILL.md']
            front = yaml.safe_load(manifest.split('---', 2)[1])
            assert front['name'] == skill.rsplit('/', 1)[-1]
        for tool in cfg.get('tools') or []:
            if isinstance(tool, dict):
                check_agent(tool['agent_tool']['config_path'], False)
    check_agent('agent.yaml', True)
    if 'eval_config.yaml' in files:
        limits = yaml.safe_load(files['eval_config.yaml'])['evaluation']
        assert 0 < limits['max_time_minutes'] <= 5
        assert 0 < limits['max_tool_calls'] <= 100
        assert 0 < limits['max_turns'] <= 250
        assert 0 < limits['timeout_seconds'] <= 300

def pack(files: dict[str, str]) -> bytes:
    validate(files)
    stream = io.BytesIO()
    with zipfile.ZipFile(stream, 'w') as archive:
        for name, content in sorted(files.items()):
            entry = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
            entry.compress_type = zipfile.ZIP_DEFLATED
            entry.create_system = 3
            entry.external_attr = 0o644 << 16
            archive.writestr(entry, content.encode('utf-8'))
    result = stream.getvalue()
    assert unpack(result) == files
    return result

candidate_error = None
try:
    candidate_bytes = pack(challenger)
except Exception as error:
    candidate_bytes = None
    candidate_error = f'{type(error).__name__}: {error}'

validate(anchor)
validate(fallback)
selected = 'v12 experimental challenger' if candidate_bytes else 'v10 scored anchor (automatic rollback)'
outputs = {'submission.zip': candidate_bytes or archive_bytes['anchor'],
           'anchor_submission.zip': archive_bytes['anchor'],
           'fallback_submission.zip': archive_bytes['fallback']}
if candidate_bytes:
    outputs['challenger_submission.zip'] = candidate_bytes
for name, data in outputs.items():
    (WORKING / name).write_bytes(data)
    assert unpack((WORKING / name).read_bytes())

card = {'version': 12, 'strategy': 'State Card', 'selected': selected,
        'anchor_public_score': 0.12, 'v11_public_score': None, 'challenger_public_score': None,
        'candidate_error': candidate_error, 'changed_files': changed,
        'budget': budget['evaluation'], 'model_seed': None,
        'model_inference': 'stochastic; no stability claim',
        'notebook_hardware': 'CPU',
        'hashes': {name: sha(data) for name, data in outputs.items()}}
(WORKING / 'experiment_card.json').write_text(json.dumps(card, indent=2), encoding='utf-8')
assert sha(outputs['anchor_submission.zip']) == EXPECTED_SHA256['anchor']
assert sha(outputs['fallback_submission.zip']) == EXPECTED_SHA256['fallback']
print('✅ FINAL STATIC VALIDATION PASSED')
print('🧪' if candidate_bytes else '🛡️', selected)
for name, data in outputs.items():
    print(f'📦 {name}: {len(data):,} bytes | SHA256 {sha(data)[:16]}')

## 4 · How to judge v12

The notebook produces `submission.zip` for a separate experimental leaderboard submission, `anchor_submission.zip` as the exact 0.12 bundle, and `fallback_submission.zip` as the scored 0.10 baseline. Run an official evaluator pilot on the same held-out FastAPI, Rich and Requests tasks for v12 and the anchor. Record solve rate, regressions, tool calls, task time, context failures and whether the analyzer was called for a second diagnosis. Promote v12 only after a completed score and runtime review show a credible gain. A worse or inconclusive result returns to the anchor. The next isolated experiment can test multi-query localization in `repo-scout`.

Thanks to Roman Rozen, Mizero Lucky Gall and the competition hosts for their public notebooks and harness documentation.